# Kilonovae Detection Rate Analysis
#### This notebook estimates the detection rate of kilonovae (optical counterparts 
#### of gravitational wave events) for the ULTRASAT mission, using simulated 
#### observations from the M4OPT scheduler.

In [ ]:
# --- Installation & Setup (Colab only) ---
# Clone our GitHub repository into the Colab environment

import sys

# Check if the env run under Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/weizmannk/EarthOrbitPlan.git
    %cd EarthOrbitPlan
    !pip install -e .

    print("Environment ready. You can now run the rest of the notebook.")

In [ ]:
import os
import sys

# Check if the env run under Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Go to the "EarthOrbitPlan/earthorbitplan/tutorials"
    os.chdir("./earthorbitplan/tutorials")

    # Check if the 'kilonovae_detection_rate.ipynb' is there
    print(os.listdir())

In [ ]:
import logging
import os
import warnings

import numpy as np
from astropy.table import QTable
from matplotlib import pyplot as plt

from earthorbitplan.probability.rate import poisson_lognormal_rate_quantiles
from earthorbitplan.utils.path import get_project_root
from earthorbitplan.workflow.area_distance import plot_area_distance
from earthorbitplan.workflow.selection_detection_rate import (
    summarize_selected_detected_events,
)

# Suppress known warnings for cleaner output

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", ".*dubious year.*")
warnings.filterwarnings(
    "ignore", "Tried to get polar motions for times after IERS data is valid.*"
)

### Setup logging 

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,
)

In [ ]:
# =====================================================================
# Analysis configurations - single source of truth for every path.
# A cell downstream selects one entry with CONFIGS["<name>"]; no events
# file or export name is ever written twice in this notebook.
# =====================================================================
root = get_project_root()

# Exports anchored on the project root, not on the kernel's cwd.
paper_dir = root / "scripts" / "paper_plots"
table_dir = paper_dir / "tables"
plot_dir = paper_dir / "plots"
table_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

# The run duration for each observing run
run_duration = 1  # years

# Naming follows what m4opt already writes into the "skygrid" column and what
# plot_area_distance derives from it: "_" separates fields, "-" lives inside a
# field value ("non-overlap"). So every artefact of one configuration shares
# the stem "<mission>_<skygrid>".
# Merger rate priors, one entry per population model -- the keys are the same
# names as the "[scenarios] pop" setting of the .ini configuration, so a run and
# its rate prior cannot drift apart.
#
# GWTC-5.0 FullPop: fiducial total CBC merger rate density
#   100 (+103, -50) Gpc^-3 yr^-1 (90% CI)  ->  5/50/95% quantiles below.
MERGER_RATE_PRIORS = {
    "fullpop": {"lo": 50.0, "mid": 100.0, "hi": 203.0},
    "pixelpop": {"lo": 39, "mid": 65, "hi": 139},
}

CONFIGS = {
    "allsky": {
        "events_file": root / "data" / "ultrasat" / "ultrasat_allsky.ecsv",
        "table_file": table_dir / "ultrasat_allsky_detection.tex",
        "figure_prefix": plot_dir / "ultrasat_allsky_obj-vs-det",
        "pop": "fullpop",
    },
    "non-overlap": {
        "events_file": root / "data" / "ultrasat" / "ultrasat_non-overlap.ecsv",
        "table_file": table_dir / "ultrasat_non-overlap_detection.tex",
        "figure_prefix": plot_dir / "ultrasat_non-overlap_obj-vs-det",
        "pop": "fullpop",
    },
    "uvex": {
        "events_file": root / "data" / "uvex" / "uvex.ecsv",
        "table_file": table_dir / "uvex_detection.tex",
        "figure_prefix": plot_dir / "uvex_obj-vs-det",
        "pop": "fullpop",
    },
}

print(f"exports -> {paper_dir}")
for name, cfg in CONFIGS.items():
    status = "found" if cfg["events_file"].exists() else "MISSING"
    print(f"  {name:12s} {status:9s} {cfg['events_file'].relative_to(root)}")

### Load simulated event data

In [ ]:
config = CONFIGS["non-overlap"]
main_table = QTable.read(config["events_file"])

config["events_file"]

### Get unique run names

In [ ]:
runs = np.unique(main_table["run"])
print(runs)

### Filter events by objective_value cutoff

In [ ]:
cutoff = main_table["cutoff"][0]
main_table = main_table[main_table["objective_value"] >= cutoff]
main_table[0:1]

### Group events by run

In [ ]:
event_tables_by_run = {run: main_table[main_table["run"] == run] for run in runs}
event_tables_by_run[runs[0]][0:1]

# Detection rates by run and source class

We estimate the expected number of **selected** and **detected** events for each observing run (IR1HLV, O5a, O5b, O5c) and source class (BNS, NSBH, All), using a **Poisson-lognormal model**

- **Selected**: events with objective value ≥ cutoff
- **Detected**: selected events weighted by detection probability (known position)
- **Credible intervals**: 90%, from the lognormal merger rate prior

---

## 1 — Merger rate prior (GWTC-5.0)

Following Petrov et al. (`PeSi2021`) and Kiendrébéogo et al. (`Kiendrebeogo_2023`), we adopt the fiducial **total CBC** merger rate density from **GWTC-5.0** (`gwtc5_population`):

$$\mathcal{R}_{\mathrm{CBC}}^{\mathrm{fid}} = 100^{+103}_{-50}\ \mathrm{Gpc^{-3}\,yr^{-1}} \qquad (90\%\ \mathrm{CI})$$

These are the **FullPop** numbers. They are declared once, in the `MERGER_RATE_PRIORS` table of the configuration cell above, keyed by the same population model name as the `[scenarios] pop` setting of the `.ini` configuration — so a run and its rate prior cannot drift apart. Each `CONFIGS` entry names its population model under `"pop"`, and the cells below look the prior up with `MERGER_RATE_PRIORS[config["pop"]]`. A PixelPop analysis needs a new entry in that table and `"pop": "pixelpop"` in its configuration.

The declared quantiles are:

| Quantile | Rate |
|---|---|
| 5% | 50 $\mathrm{Gpc^{-3}\,yr^{-1}}$ |
| 50% (median) | 100 $\mathrm{Gpc^{-3}\,yr^{-1}}$ |
| 95% | 203 $\mathrm{Gpc^{-3}\,yr^{-1}}$ |

The lognormal prior is built from the median and from the 5–95% width, $\sigma = \log(203/50)/3.2897 = 0.426$. These three numbers are mutually consistent with a lognormal to better than 1%.

## 2 — Log-expected rate per run and source class

For each run $r$ and source class $c$ (BNS, NSBH, All), the log-expected number of events is:

$$\mu_{r,c} = \log(\text{target median}) + \log(\text{run duration}) - \log(\text{effective rate}_{\text{sim},r}) + \log(N_{r,c})$$

where $N_{r,c}$ is either the number of selected events or the sum of detection probabilities for class $c$ in run $r$. Note that the effective simulation rate depends only on the run, not the class.

## 3 — Poisson-lognormal quantiles

The 5%, 50%, and 95% quantiles of the rate are computed from the Poisson-lognormal model and reported as $\mathrm{median}_{-\mathrm{lo}}^{+\mathrm{hi}}$ in the table below.

### ULTRASAT AllSky grid

In [ ]:
# Compute detection rates by run and source class (BNS, NSBH, All)

config = CONFIGS["allsky"]
prior = MERGER_RATE_PRIORS[config["pop"]]

latex_table = summarize_selected_detected_events(
    config["events_file"],
    quantiles=(0.5, 0.05, 0.95),
    merger_rate_lo=prior["lo"],
    merger_rate_mid=prior["mid"],
    merger_rate_hi=prior["hi"],
    run_duration=run_duration,
    poisson_lognormal_rate_quantiles=poisson_lognormal_rate_quantiles,
    output_file=config["table_file"],
    verbose=True,
)

### ULTRASAT Non-Overlap Sky grid

In [ ]:
# Compute detection rates by run and source class (BNS, NSBH, All)

config = CONFIGS["non-overlap"]
prior = MERGER_RATE_PRIORS[config["pop"]]

latex_table = summarize_selected_detected_events(
    config["events_file"],
    quantiles=(0.5, 0.05, 0.95),
    merger_rate_lo=prior["lo"],
    merger_rate_mid=prior["mid"],
    merger_rate_hi=prior["hi"],
    run_duration=run_duration,
    poisson_lognormal_rate_quantiles=poisson_lognormal_rate_quantiles,
    output_file=config["table_file"],
    verbose=True,
)

## UVEX

In [ ]:
# Compute detection rates by run and source class (BNS, NSBH, All)

config = CONFIGS["uvex"]
prior = MERGER_RATE_PRIORS[config["pop"]]

latex_table = summarize_selected_detected_events(
    config["events_file"],
    quantiles=(0.5, 0.05, 0.95),
    merger_rate_lo=prior["lo"],
    merger_rate_mid=prior["mid"],
    merger_rate_hi=prior["hi"],
    run_duration=run_duration,
    poisson_lognormal_rate_quantiles=poisson_lognormal_rate_quantiles,
    output_file=config["table_file"],
    verbose=True,
)

# Area–Distance detection efficiency plot

#### Scatter plot of triggered GW follow-up events in the (distance, sky area) plane.
#### Each star encodes the **objective value** (size) and **detection probability** (color),
#### with marginal histograms and theoretical limit boundaries ($d^{-4}$, max area, max distance).

## ULTRASAT with AllSky grid

In [ ]:
config = CONFIGS["allsky"]
plot_area_distance(config["events_file"], outdir=plot_dir, show=True)

# ULTRASAT with non-overlap Skygrid

In [ ]:
config = CONFIGS["non-overlap"]
plot_area_distance(config["events_file"], outdir=plot_dir, show=True)

In [ ]:
def plot_objective_vs_detection(table, cutoff, run, figurename="allsky_obj-vs-det"):
    """
    Objective vs Detection with area as marker size.
    - X: objective_value
    - Y: detection_probability
    - Color: distance
    - Size: sky area (90% credible region)
    """

    triggered_table = table[table["objective_value"] >= cutoff]

    # ===================================================================
    # Extract data
    # ===================================================================
    obj_val = triggered_table["objective_value"]
    det_prob = triggered_table["detection_probability_known_position"]
    distance = triggered_table["distance"]
    area = triggered_table["area(90)"]

    # ===================================================================
    # Normalize area for marker size
    # ===================================================================
    # Small area => small marker, large area => large marker
    # Scale between 50 and 1000 for visibility
    size_min, size_max = 50, 2000
    area_normalized = (area - area.min()) / (area.max() - area.min())
    sizes = size_min + area_normalized * (size_max - size_min)

    # ===================================================================
    # Create figure
    # ===================================================================
    _, ax = plt.subplots(figsize=(10, 8))

    scatter = ax.scatter(
        obj_val,
        det_prob,
        s=sizes,  # <= Size proportional to area
        c=distance,
        cmap="viridis",
        alpha=0.7,  # Slightly more transparent for overlapping
        edgecolors="black",
        linewidth=0.8,
    )

    # Reference lines
    ax.axvline(
        cutoff,
        color="red",
        linestyle="--",
        linewidth=2.5,
        label=f"Cutoff = {cutoff:.2f}",
    )
    ax.axhline(0.5, color="orange", linestyle="--", linewidth=2, label="50% detection")
    ax.plot([0, 1], [0, 1], "k:", linewidth=1.5, alpha=0.3, label="Perfect correlation")

    # Labels
    ax.set_xlabel("Objective Value", fontsize=13, weight="bold")
    ax.set_ylabel("Detection Probability", fontsize=13, weight="bold")
    ax.set_title(
        f"{run}\nMarker size ∝ Sky localization area", fontsize=15, weight="bold"
    )
    ax.set_xlim(cutoff - 0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

    # Legend
    ax.legend(loc="lower right", fontsize=10, framealpha=0.95)

    # ===================================================================
    # Colorbar for distance
    # ===================================================================
    plt.colorbar(scatter, ax=ax, label="Distance (Mpc)")

    # ===================================================================
    # Size legend for area
    # ===================================================================
    # Create manual legend for marker sizes
    from matplotlib.lines import Line2D

    # Representative sizes (small, medium, large)
    area_percentiles = np.percentile(area, [10, 50, 90])
    size_percentiles = np.percentile(sizes, [10, 50, 90])

    legend_elements = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=np.sqrt(size_percentiles[0] / 4),
            label=rf"{area_percentiles[0]:.0f} deg$^2$",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=np.sqrt(size_percentiles[1] / 4),
            label=rf"{area_percentiles[1]:.0f} deg$^2$",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=np.sqrt(size_percentiles[2] / 4),
            label=rf"{area_percentiles[2]:.0f} deg$^2$",
        ),
    ]

    ax.legend(
        handles=legend_elements,
        loc="lower right",
        title="Sky Area (90%)",
        fontsize=9,
        framealpha=0.95,
        title_fontsize=10,
    )
    output_file = f"{figurename}_{run}.pdf"

    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    plt.show()

In [ ]:
config = CONFIGS["non-overlap"]
main_table = QTable.read(config["events_file"])

cutoff = main_table["cutoff"][0]
run = "O5c"

table = main_table[main_table["run"] == run]

plot_objective_vs_detection(table, cutoff, run, figurename=config["figure_prefix"])

## UVEX Schedule

In [ ]:
config = CONFIGS["uvex"]
plot_area_distance(config["events_file"], outdir=plot_dir, show=True)

In [ ]:
config = CONFIGS["uvex"]
main_table = QTable.read(config["events_file"])

cutoff = main_table["cutoff"][0]
run = "O5c"

table = main_table[main_table["run"] == run]

plot_objective_vs_detection(table, cutoff, run, figurename=config["figure_prefix"])